In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
df = pd.read_excel("Dataset Wisata.xlsx")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Dataset Wisata.xlsx'

In [ ]:
# Bersihkan rating (karena pakai koma)
df['Rating'] = df['Rating'].astype(str).str.replace(',', '.')
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')

df['Rating'] = df['Rating'].fillna(0)

df.head()

,No,Nama Tempat,Kategori,Jam Buka,Jam Tutup,Durasi disarankan (jam),GPS Location,HTM Min Domestic,HTM Max Domestic,HTM Min Mancanegara,HTM Max Mancanegara,Moda Transportasi,Keterangan,Sumber,Status,Rating
0,1,Waduk Tirto Agro -Glenmore,Wisata Alam,07:30:00,16:00:00,2,"-8.2788264,113.9324687",10000,10000,10000,10000,"Mobil, Sepeda Motor",tutup,https://id.trip.com/travel-guide/attraction/ba...,Tutup,0.0
1,2,Penangkaran Banteng Baluran,Wisata Edukasi,07:30:00,16:00:00,4,"-7.83097579646207, 114.38752901123405",21000,31000,150000,225000,"Mobil, Sepeda Motor, Bus",bisa sekaligus dengan situs lainnya yang ada d...,https://www.yukbanyuwangi.co.id/harga-tiket-ta...,Tutup,0.0
2,3,Kebun Kopi Kemiren,Wisata Edukasi,08:00:00,17:00:00,3,"-8.201449434118924, 114.32217474598616",0,0,0,0,"Mobil, Sepeda Motor",Minggu buka pukul 7-12 siang,NaN,Tutup,4.7
3,4,Sanggar Genjah Arum,Wisata Budaya,08:00:00,16:00:00,3,"-8.2018766,114.298193",20000,450000,20000,450000,"mobil, sepeda motor, bus",NaN,https://atourin.com/destination/banyuwangi/san...,Tutup,4.7
4,7,Bukit Mondoleko,Wisata Alam,05:00:00,17:00:00,2,"-8.235249255000987, 114.16452966653851",10000,10000,10000,10000,"Mobil, Sepeda Motor",sudah tidak aktif,https://atourin.com/destination/banyuwangi/buk...,Tutup,4.0


In [ ]:
ratings = pd.DataFrame({
    'user_id': np.random.randint(1, 15, 80),
    'wisata': np.random.choice(df['Nama Tempat'], 80),
    'rating': np.random.randint(3, 6, 80)
})

ratings.head()

,user_id,wisata,rating
0,3,Masjid Cheng Ho,4
1,7,Sun Osing,4
2,13,Green island,3
3,2,Banyuwangi Park,3
4,7,Klenteng Hoo Tong Bio,4


In [ ]:
saved = pd.DataFrame({
    'user_id': np.random.randint(1, 15, 40),
    'wisata': np.random.choice(df['Nama Tempat'], 40),
    'saved': 1
})

saved.head()

,user_id,wisata,saved
0,4,Situs Kawitan Alas Purwo,1
1,7,Pulau Bedil,1
2,11,Pulau Menjangan,1
3,7,Gua Maria Curahjati,1
4,9,pantai pulau santen,1


In [ ]:
interaction = pd.merge(
    ratings,
    saved,
    on=['user_id','wisata'],
    how='outer'
).fillna(0)

interaction['score'] = interaction['rating'] + interaction['saved']

interaction.head()

,user_id,wisata,rating,saved,score
0,1,Air Terjun Kalibendo,3.0,0.0,3.0
1,1,Air Terjun Kedung Angin,5.0,0.0,5.0
2,1,Bukit Sewu Sambang,0.0,1.0,1.0
3,1,Kebun Naga dan Jeruk Pesanggaran,0.0,1.0,1.0
4,1,Masjid Agung Baiturahman,3.0,0.0,3.0


In [ ]:
user_item_matrix = interaction.pivot_table(
    index='user_id',
    columns='wisata',
    values='rating'
).fillna(0)

user_item_matrix.head()

wisata,Agro Wisata Gunug Gumitir,Air Terjun Giri Asih,Air Terjun Kalibendo,Air Terjun Kedung Angin,Air Terjun Legomoro,Air Terjun Lider,Air Terjun Selendang Arum,Air Terjun Telunjuk Raung,Air Terjun Temcor,Air Terjun Tirto Kemanten,...,Teluk Hijau,Waduk Bajulmati,Waduk Tirto Agro -Glenmore,capas,doeson kakoe,kali sawah adventure,pantai Trianggulasi,pantai pancur alas purwo,pantai pulau santen,wisata goa sodong
user_id,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,3.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,4.0,0.0,0.0,0.000000,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,3.0,0.0,3.666667,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,...,5.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,4.0,0.000000,0.0,0.0,0.0


In [ ]:
similarity = cosine_similarity(user_item_matrix)

similarity_df = pd.DataFrame(
    similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

similarity_df.head(10)

user_id,1,2,3,4,5,6,7,8,9,10,11,12,13,14
user_id,,,,,,,,,,,,,,
1,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.090221,0.000000,0.219971,0.271755,0.000000,0.0
2,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.290021,0.000000,0.309356,0.000000,0.000000,0.000000,0.000000,0.0
3,0.000000,0.000000,1.000000,0.000000,0.000000,0.083576,0.000000,0.000000,0.000000,0.000000,0.231611,0.128761,0.000000,0.0
4,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.140681,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
5,0.000000,0.000000,0.000000,0.000000,1.000000,0.127418,0.000000,0.191127,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
6,0.000000,0.000000,0.083576,0.000000,0.127418,1.000000,0.296001,0.050000,0.078934,0.000000,0.000000,0.064194,0.106299,0.0
7,0.000000,0.290021,0.000000,0.140681,0.000000,0.296001,1.000000,0.126294,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
8,0.000000,0.000000,0.000000,0.000000,0.191127,0.050000,0.126294,1.000000,0.126294,0.186772,0.000000,0.000000,0.000000,0.0
9,0.090221,0.309356,0.000000,0.000000,0.000000,0.078934,0.000000,0.126294,1.000000,0.000000,0.205076,0.000000,0.000000,0.0


In [ ]:
def get_cf_wisata(user_id):

    similar_users = similarity_df[user_id]\
        .sort_values(ascending=False)[1:6]

    similar_users_index = similar_users.index

    rekomendasi = interaction[
        interaction['user_id'].isin(similar_users_index)
    ]

    rekomendasi = rekomendasi.sort_values(
        by='rating',
        ascending=False
    )

    return rekomendasi['wisata'].unique()

In [ ]:
def content_based_filter(
    wisata_list,
    kategori,
    budget_min,
    budget_max,
    rating_min
):

    filtered = df[
        (df['Nama Tempat'].isin(wisata_list)) &
        (df['Kategori'].isin(kategori)) &
        (df['HTM Min Domestic'] >= budget_min) &
        (df['HTM Max Domestic'] <= budget_max) &
        (df['Rating'] >= rating_min)
    ]

    return filtered

In [ ]:
def hybrid_recommendation(
    user_id,
    kategori,
    budget_min,
    budget_max,
    rating_min
):

    cf_result = get_cf_wisata(user_id)

    # # fallback jika CF sedikit
    if len(cf_result) < 5:
        cf_result = df['Nama Tempat']

    final = content_based_filter(
        cf_result,
        kategori,
        budget_min,
        budget_max,
        rating_min
    )

    final = final.sort_values(
        by='Rating',
        ascending=False
    )

    return final

In [ ]:
hasil = hybrid_recommendation(
    user_id=1,
    kategori=['Wisata Alam', 'Wisata Religi', 'Wisata Edukasi'],
    budget_min=0,
    budget_max=20000,
    rating_min=4
)

print(hasil[['Nama Tempat','Kategori', 'HTM Min Domestic', 'HTM Max Domestic', 'HTM Min Mancanegara', 'HTM Max Mancanegara', 'Rating']])

                        Nama Tempat        Kategori  HTM Min Domestic  \
38                   Pantai Mustika     Wisata Alam              5000   
49                      Pantai Boom     Wisata Alam              7500   
54             Bangsring Underwater     Wisata Alam              5000   
33        Air Terjun Telunjuk Raung     Wisata Alam             10000   
31             Air Terjun Kalibendo     Wisata Alam              5000   
37                Pantai Wedi Ireng     Wisata Alam              7500   
74            Gumuk Kancil Glenmore   Wisata Religi                 0   
77                  Masjid Cheng Ho   Wisata Religi                 0   
13                 Air Terjun Lider     Wisata Alam             10000   
36                 Pantai Rajegwesi     Wisata Alam              5000   
73            Makam Datuk Abdurahim   Wisata Religi                 0   
51                 Pantai Watudodol     Wisata Alam              7500   
66  Penangkaran Penyu Pantai Cemara  Wisata Edukasi

In [ ]:
cf_result = get_cf_wisata(2)
cf_result

array(['Air Terjun Kedung Angin', 'Gumuk Kancil Glenmore', 'Pelangi Sari',
       'Penangkaran Penyu Pantai Cemara', 'Waduk Tirto Agro -Glenmore',
       'Teluk Hijau', 'Banyuwangi Park', 'Waduk Bajulmati', 'Pantai Boom',
       'Masjid Cheng Ho', 'capas ', 'Air Terjun Legomoro',
       'pantai Trianggulasi', 'wisata goa sodong',
       'Klenteng Hoo Tong Bio', 'Pantai Watudodol', 'Puncak Asmoro',
       'Agro Wisata Gunug Gumitir', 'Sun Osing', 'Bangsring Underwater',
       'Situs Kendeng Lembu', 'Pantai Rajegwesi', 'Air Terjun Kalibendo',
       'Pasar blambangan', 'doeson kakoe', 'Pinus reborn (Suwis)',
       'Masjid Agung Baiturahman', 'Situs Kawitan Alas Purwo',
       'Kebun Naga dan Jeruk Pesanggaran', 'Bukit Sewu Sambang',
       'Pantai Wedi Ireng', 'Taman Gandrung terakota',
       'Air Terjun Selendang Arum', 'Perkebunan Kaliklatak',
       'Pantai Cemara - Kawang', 'Gua Maria Curahjati', 'Pulau Bedil',
       'Pantai Sukamade', 'Hutan Mangrove Trail Baluran',
       'Air 